# Single-GPU process-kill recovery

Select Runtime > Change runtime type > T4 GPU if free compute is available. Run cells in order. This experiment kills one spawned training worker with SIGKILL after optimizer step 2, restores durable step 1 in a new worker, and compares against an independent control.

This is one paused boundary, not runtime deletion, power loss, multi-GPU recovery, production reliability or model-quality evidence. No CPU substitute, paid compute, Drive mount or public service is used. Download the compact report before the temporary runtime disappears.


In [ ]:
import hashlib
import json
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path


def checked(arguments, *, cwd=None, env=None, timeout=180):
    try:
        result = subprocess.run(
            arguments, cwd=cwd, env=env, check=True, capture_output=True, text=True, timeout=timeout
        )
    except subprocess.CalledProcessError as error:
        print((error.stdout or "")[-8000:])
        print((error.stderr or "")[-2000:])
        raise
    print(result.stdout[-8000:])
    if result.stderr:
        print(result.stderr[-2000:])
    return result.stdout


hardware = checked(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total,compute_cap",
        "--format=csv,noheader",
    ]
)
assert hardware.strip(), "No NVIDIA GPU observed"
IMPLEMENTATION_REVISION = "65d280888e0f34aa9dc7107f8197dcef7f48fa34"
assert re.fullmatch(r"[0-9a-f]{40}", IMPLEMENTATION_REVISION)
checked([sys.executable, "-m", "pip", "install", "uv==0.9.7"], timeout=900)
LAB_ROOT = Path(tempfile.mkdtemp(prefix="fttl-process-", dir="/content")) / "lab"
checked(
    [
        "git",
        "clone",
        "https://github.com/ajaykr0905/fault-tolerant-transformer-lab.git",
        str(LAB_ROOT),
    ]
)
checked(["git", "checkout", "--detach", IMPLEMENTATION_REVISION], cwd=LAB_ROOT)
assert checked(["git", "rev-parse", "HEAD"], cwd=LAB_ROOT).strip() == IMPLEMENTATION_REVISION
assert not checked(["git", "status", "--porcelain"], cwd=LAB_ROOT).strip()
RUN_ENV = dict(os.environ, CUBLAS_WORKSPACE_CONFIG=":4096:8")
checked(
    [sys.executable, "-m", "uv", "sync", "--frozen", "--extra", "test", "--python", "3.12"],
    cwd=LAB_ROOT,
    env=RUN_ENV,
    timeout=900,
)
PYTHON = str(LAB_ROOT / ".venv/bin/python")

In [ ]:
# Each target runs in a fresh process. A skipped GPU test is not validation.
for target in [
    "tests/test_cuda_process_recovery.py",
    "tests/test_cuda_runtime.py",
    "tests/test_cuda_recovery.py",
    "tests/test_checkpoint_integrity.py::test_real_cuda_rng_checkpoint_is_weights_only_safe_and_cpu_loadable",
]:
    output = checked(
        [PYTHON, "-m", "pytest", "-q", "-rs", target], cwd=LAB_ROOT, env=RUN_ENV, timeout=600
    )
    assert "skipped" not in output, "Real CUDA regression tests were skipped"

In [ ]:
DATASET = LAB_ROOT / ".cache/fttl/peps-v1"
checked(
    [
        PYTHON,
        "-m",
        "fttl.prepare_peps_cli",
        "--cache-dir",
        ".cache/downloads",
        "--output",
        str(DATASET),
    ],
    cwd=LAB_ROOT,
    env=RUN_ENV,
)
OUTPUT = LAB_ROOT / "artifacts/colab-cuda-process-001"
assert not OUTPUT.exists(), "Use a fresh output path"
checked(
    [
        PYTHON,
        "-m",
        "fttl.cuda_process_recovery",
        "--config",
        "configs/peps-cpu.json",
        "--dataset-manifest",
        str(DATASET / "manifest.json"),
        "--output",
        str(OUTPUT),
        "--timeout-seconds",
        "120",
    ],
    cwd=LAB_ROOT,
    env=RUN_ENV,
    timeout=600,
)
REPORT_PATH = OUTPUT / "cuda-process-recovery-report.json"
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
assert report["schema"] == "CudaProcessRecoveryReportV1"
assert report["code_revision"] == IMPLEMENTATION_REVISION
assert report["execution"]["device"] == "cuda:0"
assert report["kill_signal"] == "SIGKILL" and report["interrupted_exitcode"] == -9
assert len(set(report["worker_pids"].values())) == 4
assert report["selected_step"] == report["durable_step_before_kill"] == 1
assert report["failed_batch_id"] == report["replayed_batch_id"]
assert report["failed_sample_ids"] == report["replayed_sample_ids"]
assert report["exact_equality"] is True and all(report["equality"].values())
assert report["steps"] == 6 and report["tokens_seen"] == 384
print(json.dumps(report, indent=2, sort_keys=True))
print("Evidence SHA256:", hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest())

In [ ]:
# Export only the small report. Keep trusted-local binary state out of Git.
from google.colab import files

files.download(str(REPORT_PATH))